In [1]:
import sys 
sys.path.append("../src")

from openf1_client import get

In [ ]:
import pandas as pd

ENDPOINTS = ["laps", "stints", "race_control", "weather"]

fp2 = get("sessions", use_cache = False, year = 2026, session_name = "Practice 2")
races = get("sessions", use_cache = False, year = 2026, session_name = "Race")
sessions = pd.concat([fp2, races])

summary = []    #list of dictionaries

for _,s in sessions.iterrows():
    key = s["session_key"]
    print("Downloading", s["location"], s["session_name"], key)

    n_laps = 0
    for endpoint in ENDPOINTS:
        df = get(endpoint, session_key = key)
        if endpoint == "laps":
            n_laps = len(df)
    summary.append({
        "location": s["location"],
        "session": s["session_name"],
        "session_key": key,
        "laps": n_laps,
    })


summary = pd.DataFrame(summary)
summary


session
Race          21
Practice 2    15
Name: count, dtype: int64

In [15]:
with_data = summary[summary["laps"] > 0]
with_data["session"].value_counts()

session
Race          21
Practice 2    15
Name: count, dtype: int64

-**Coverage:** OpenF1 has lap data for 21 races and 15 FP2 sessions of the 2026 season (up to Las Vegas). Sakhir and Jeddah were cancelled, so they exist in the API but have no data. Lusail and Yas Marina have not happened yet.

-**Why fewer FP2 than races:** sprint weekends have only one practice session (FP1). Sprint qualifying replaces FP2, so those weekends cannot be used by this model.

-**Usable pairs:** every weekend with FP2 data also has a race, which gives 15 FP2 + race pairs. Las Vegas FP2 was fully wet (INTERMEDIATE/WET tyres), so **14 pairs** are usable for dry race pace.

-**Caching:** all raw responses are stored locally in `data/raw/openf1/` (not committed). A re-run takes seconds instead of minutes, and the analysis no longer depends on the API being available.
